[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-multivariate-gibbs.ipynb)

# Multivariate Gaussian Distribution & Gibbs Sampling

*AIBits Academy · Machine Learning End To End · ⚠ Advanced Topic*

The first of two MCMC (Markov Chain Monte Carlo) techniques: when you can't sample a joint distribution directly, sample one variable at a time, conditioned on the others, and let the chain converge to the real thing.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

> **⚠ Why This Page Is Marked "Advanced"**
>
> Gibbs sampling is the first genuinely *iterative* sampling technique in this course — instead of drawing an independent sample in one shot (like every distribution used so far), each new sample depends on the previous one, forming a Markov chain whose long-run behaviour matches the distribution you actually want. Understanding *why* that works, and how to check it's working, is the core skill this page and the next build.

## The Multivariate Gaussian Distribution

The (univariate) normal distribution generalises directly to multiple, potentially correlated variables. A **mean vector** μ replaces the single mean, and a **covariance matrix** Σ replaces the single variance — its diagonal holds each variable's own variance, and its off-diagonal entries capture how variables move together:

$$f(x) = (2\pi)^{-k/2}|\Sigma|^{-1/2}\exp\!\left(-\tfrac{1}{2}(x-\mu)^\mathsf{T}\Sigma^{-1}(x-\mu)\right) \qquad k = \text{number of variables}$$

> **📊 Prerequisite refresher**
>
> Σ⁻¹ (the covariance matrix's inverse) and |Σ| (its determinant) are exactly the matrix operations covered in Linear Algebra for ML — the same eigendecomposition machinery used there to find a covariance matrix's principal directions is what determines the "shape" (the elliptical contours) of a multivariate Gaussian.

## Gibbs Sampler — Sampling When You Can't Sample Directly

Sometimes drawing directly from a joint distribution is hard, but drawing from each variable's **conditional** distribution — holding every other variable fixed at its current value — is easy. The Gibbs sampler exploits exactly this: cycle through each variable, resampling it from its conditional given the current values of all the others, and repeat. After enough iterations, the sequence of samples converges to the true joint distribution.

- Start at some initial point (x₀, y₀) — the starting values barely matter given enough iterations

- Sample xₜ₊₁ from P(x | y = yₜ) — x's conditional distribution, given the current y

- Sample yₜ₊₁ from P(y | x = xₜ₊₁) — y's conditional distribution, given the just-updated x

- Repeat thousands of times; discard an initial "burn-in" period before the chain has converged

For a bivariate Gaussian specifically, both conditionals are themselves simple univariate normals with a known closed form — which is exactly what makes this a clean first example of the technique:

$$x\mid y \sim \mathcal{N}\!\left(\mu_x + \dfrac{\Sigma_{xy}}{\sigma_y^2}(y-\mu_y),\ \ \sigma_x^2(1-\rho^2)\right) \qquad \text{and symmetrically for } y\mid x$$

## Worked Example — Two Correlated Business Metrics

Order value and delivery lead time at a Swiggy branch tend to move together — larger orders take longer to prepare and deliver. Modeling them as a bivariate Gaussian with a known true mean, variance and correlation, then *recovering* those same parameters purely from Gibbs-sampled draws:

In [ ]:
import numpy as np

# True parameters: order value (Rs) and delivery lead time (minutes)
mu = np.array([450.0, 35.0])
sigma_x, sigma_y = 120.0, 8.0
rho = 0.6
cov_xy = rho * sigma_x * sigma_y
Sigma = np.array([[sigma_x**2, cov_xy], [cov_xy, sigma_y**2]])

def gibbs_sampler_bivariate(mu, Sigma, num_samples, burn_in=1000):
    mu_x, mu_y = mu
    var_x, var_y, cov = Sigma[0,0], Sigma[1,1], Sigma[0,1]
    rho_local = cov / np.sqrt(var_x*var_y)
    x, y = 0.0, 0.0
    samples = np.zeros((num_samples, 2))
    for i in range(num_samples):
        mean_x = mu_x + (cov/var_y)*(y - mu_y)
        x = np.random.normal(mean_x, np.sqrt(var_x*(1-rho_local**2)))
        mean_y = mu_y + (cov/var_x)*(x - mu_x)
        y = np.random.normal(mean_y, np.sqrt(var_y*(1-rho_local**2)))
        samples[i] = [x, y]
    return samples[burn_in:]

samples = gibbs_sampler_bivariate(mu, Sigma, num_samples=10000, burn_in=1000)
print(f"Recovered mean: order_value=Rs.{samples[:,0].mean():.2f}, lead_time={samples[:,1].mean():.2f} min")
print(f"Recovered correlation: {np.corrcoef(samples.T)[0,1]:.4f}   (true rho={rho})")

From 9,000 post-burn-in samples, the Gibbs sampler recovers a mean and correlation essentially indistinguishable from the true generating values (₹451.08 vs. ₹450, correlation 0.603 vs. 0.6) — despite never being given the joint distribution directly, only the two conditionals.

## Checking Convergence — Autocorrelation

Because each Gibbs sample depends on the one before it, consecutive samples are correlated — the chain needs to "forget" its starting point before it faithfully represents the target distribution. The autocorrelation function (ACF) measures how much that dependence persists across a growing gap (lag) between samples:

Autocorrelation drops from a moderate 0.36 at lag 1 to essentially zero by lag 10-20 — the chain "forgets" its recent past reasonably quickly, a healthy sign of good mixing. A chain whose ACF stayed high even at lag 50+ would need far more iterations (or thinning — keeping only every k-th sample) before its samples could be trusted as representative.

## Try It — Watch the Gibbs Sampler Build a Joint Distribution

The exact same true parameters as the code above. Each click draws one real Gibbs sample — alternately updating order value given the current lead time, then lead time given the new order value — and adds it to the live scatter plot.

> **💡 Gibbs Sampling's One Limitation**
>
> Gibbs sampling needs every variable's conditional distribution to have a known, sampleable form — which was true here only because the multivariate Gaussian's conditionals are themselves Gaussian. Many real Bayesian models don't have this convenient property. The next page's technique, Metropolis-Hastings, works even when no conditional distribution has a clean closed form.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · A conditional normal

For a bivariate normal, `X | Y=y` is normal with mean `mu_x + rho*(sx/sy)*(y - mu_y)` and standard deviation `sx*sqrt(1 - rho**2)`. Write `cond_x(y)` returning `(mean, sd)` for the lesson's order-value (x) and lead-time (y) parameters.

In [ ]:
import numpy as np
mu_x, mu_y, sx, sy, rho = 450.0, 35.0, 120.0, 8.0, 0.6
def cond_x(y):
    pass   # TODO


In [ ]:
try:
    m, s = cond_x(43.0)
    check("mean", abs(m - (450 + 0.6 * 15 * 8)) < 1e-9)
    check("sd", abs(s - 96.0) < 1e-9)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
mu_x, mu_y, sx, sy, rho = 450.0, 35.0, 120.0, 8.0, 0.6
def cond_x(y):
    return mu_x + rho * (sx / sy) * (y - mu_y), sx * np.sqrt(1 - rho ** 2)

```

</details>

### Exercise 2 · Medium · A Gibbs sampler

Write `gibbs(n, seed=0)` that alternates drawing `x | y` and `y | x` for `n` iterations (start at the means) and returns an `n × 2` array. After discarding the first 500 draws, the sample correlation should be about 0.6.

In [ ]:
import numpy as np
def cond_y(x):
    return mu_y + rho * (sy / sx) * (x - mu_x), sy * np.sqrt(1 - rho ** 2)
def gibbs(n, seed=0):
    pass   # TODO


In [ ]:
try:
    out = gibbs(6000)[500:]
    check("shape", out.shape == (5500, 2))
    check("correlation near 0.6", abs(np.corrcoef(out.T)[0, 1] - 0.6) < 0.05)
    check("means near the truth", abs(out[:, 0].mean() - 450) < 15 and abs(out[:, 1].mean() - 35) < 1)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def cond_y(x):
    return mu_y + rho * (sy / sx) * (x - mu_x), sy * np.sqrt(1 - rho ** 2)
def gibbs(n, seed=0):
    rng = np.random.default_rng(seed)
    x, y = mu_x, mu_y
    out = np.empty((n, 2))
    for i in range(n):
        m, s = cond_x(y); x = rng.normal(m, s)
        m, s = cond_y(x); y = rng.normal(m, s)
        out[i] = x, y
    return out

```

</details>

### Exercise 3 · Stretch · Autocorrelation and thinning

Consecutive Gibbs draws are correlated. Compute the lag-1 autocorrelation of the `x` chain (`ac1`), then keep every 10th draw and compute the autocorrelation of the thinned chain (`ac_thin`). Thinning should reduce it.

In [ ]:
ac1 = ac_thin = None   # TODO (reuse gibbs)


In [ ]:
try:
    check("chain is autocorrelated", ac1 > 0.05)
    check("thinning reduces it", abs(ac_thin) < ac1)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
chain = gibbs(6000)[500:, 0]
ac = lambda v: np.corrcoef(v[:-1], v[1:])[0, 1]
ac1 = ac(chain)
ac_thin = ac(chain[::10])

```

Correlated draws carry less information than independent ones; effective sample size is smaller than the raw count.

</details>

---
*Back to the course: **Machine Learning End To End → Multivariate Gaussian Distribution & Gibbs Sampling**.*